In [2]:
import pandas as pd
import numpy as np
df= pd.read_excel("Online Retail.xlsx")
print("Orginal record:",len(df))
  

Orginal record: 541909


In [9]:
df=df.dropna(subset=("CustomerID"))
df=df[~df["InvoiceNo"].astype(str).str.startswith("c")]
df=df[(df["Quantity"]>0)&(df["UnitPrice"]>0)]
df["TotalAmount"]=df["Quantity"]*df["UnitPrice"]
    


In [10]:
Customer=df.groupby("CustomerID").agg(
    Frequency=("InvoiceNo","nunique"),
        TotalQuantity=("Quantity","sum"),
        UniqueProducts=("StockCode","nunique"),
        Monetary=("TotalAmount","sum")
).reset_index()

Customer["AvgBasketValue"]=(
    Customer["Monetary"]/Customer["Frequency"])


In [11]:
Customer["Segment"]=pd.qcut(
    Customer["Monetary"],
    q=3,
labels=["Low","Medium","High"]
)
print("\nCustomer Segments:")
print(Customer["Segment"].value_counts())


Customer Segments:
Segment
Low       1446
Medium    1446
High      1446
Name: count, dtype: int64


In [12]:
features=["Frequency","TotalQuantity","UniqueProducts","Monetary","AvgBasketValue"]
for feature in features:
    Customer[feature]=pd.cut(
        Customer[feature],
        bins=3,
        labels=["Low","Medium","High"],
        include_lowest=True
    )

In [13]:
def entropy(data):
    counts=data["Segment"].value_counts()
    total=len(data)
    ent=0
    for count in counts:
        p=count/total
        if p>0:
            ent -=p*np.log2(p)
            return ent

total_entropy=entropy(Customer) 
print("\nOverall Entropy=",round(total_entropy,4))

      
      


Overall Entropy= 0.5283


In [17]:
results=[]
for feature in features:
    Weighted_entropy=0
    for value in Customer [feature].dropna().unique():
        subset=Customer[Customer[feature]==value]
        subset_entropy=entropy(subset)
        Weight=len(subset)/len(Customer)
        Weighted_entropy
      